# DEE — Test cosmológico grande con restricciones del modelo

**Autor:** Juan Pablo Bruschi (con Claude)

**Objetivo:** Determinar el escalado de la entropía de entrelazamiento del sustrato DEE en régimen mesoscópico (no UV) con N suficientemente grande para extracción confiable. De ese escalado se deriva la predicción de w cosmológico.

## Restricciones físicas del modelo DEE que respeta este notebook

1. **Régimen mesoscópico**: Sub-dominios con suficientes nodos para coarse-graining φ_i → φ(x). Mínimo: ℓ tal que N_inside ≥ 50 (esto es N≥500 efectivos por subdominio).
2. **Convergencia con N**: Sweep de N ∈ {1000, 2000, 4000, 8000} para extrapolación al continuo (siguiendo SIM 9d que usó hasta N=10648).
3. **Kernel 1/d² preservado**: derivación dimensional α=d-1=2.
4. **Separación UV/IR explícita**: medimos en bandas de escala separadas, no mezclando.
5. **Funcional físico F = N·var(κ_Ollivier)**: el de SIM 18/19 (no proxies).
6. **Validación inicial**: cristal cúbico ideal y estado térmico como controles.

## Tests que ejecuta

**Test 1**: Entropía de von Neumann por matriz de covarianza (Peschel-Eisert) en bandas de escala separadas (UV, mesoscópica, IR), con sweep de N.

**Test 2**: Escalado de aristas que cruzan bordes en bandas de escala, con sweep de N.

**Test 3**: w cosmológico vía variación de volumen (E_zp(V)) para tres regímenes:
- (a) cristal homogéneo (control, debería dar w=+0.07)
- (b) configuración glassy generada por annealing
- (c) cristal con anharmonicidad φ⁴ agregada

**Test 4**: Convergencia al continuo: extrapolación N→∞ de los exponentes medidos.

## Tiempo estimado

~6-10 horas en Colab Pro (T4 GPU o CPU). Ejecutable durante la noche.

## 0. Setup e imports

In [ ]:
# Setup automático: el notebook se ejecuta de principio a fin sin intervención.
# Al final descarga automáticamente los resultados.

import os
import sys
import time
import json
import pickle
import warnings
import numpy as np
from scipy.linalg import eigh
from scipy.spatial import KDTree
from scipy.stats import linregress, ttest_ind, ttest_1samp
from scipy.optimize import curve_fit
import matplotlib
matplotlib.use('Agg')  # backend no interactivo - importante para correr sin display
import matplotlib.pyplot as plt
from collections import defaultdict

warnings.filterwarnings('ignore')

# Crear directorio de output
OUTPUT_DIR = '/content/dee_cosmologia_resultados'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figuras', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/datos', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/logs', exist_ok=True)

# Log file: todo lo que se imprime se guarda
LOG_FILE = f'{OUTPUT_DIR}/logs/ejecucion.log'
log_handle = open(LOG_FILE, 'w')

def log(msg):
    """Imprime y guarda en log simultáneamente."""
    print(msg)
    log_handle.write(msg + '\n')
    log_handle.flush()

log("="*70)
log("DEE COSMOLOGÍA - TEST GRANDE EN RÉGIMEN MESOSCÓPICO")
log("="*70)
log(f"Inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
log(f"Directorio de salida: {OUTPUT_DIR}")
log("")

# === Configuración keep-alive ===
# Colab desconecta sesiones inactivas. Esto previene desconexión.
from IPython.display import display, Javascript
try:
    display(Javascript('''
    function ClickConnect(){
        console.log("Keep-alive ping");
        document.querySelector("colab-connect-button").shadowRoot
                .querySelector("#connect").click();
    }
    setInterval(ClickConnect, 60000);
    '''))
    log("Keep-alive activo (ping cada 60s)")
except Exception:
    log("Keep-alive no pudo activarse (estás corriendo localmente?)")

# === Estimación de tiempo ===
log("\nESTIMACIÓN DE TIEMPO:")
log("  N=1000: ~5-10 min")
log("  N=2000: ~20-40 min")
log("  N=4000: ~1.5-3 horas")
log("  N=8000: ~5-8 horas")
log("  Tests 3+4: ~30-60 min adicionales")
log("  Tests 5+6+7 (anharmonicidad): ~30-45 min adicionales")
log("  TOTAL ESTIMADO: 8-13 horas")
log("")
log("Si necesitás bajar el tiempo, en la próxima celda comentá N=8000 (línea N_values).")

## 1. Funciones core del modelo DEE

In [ ]:
# ============================================================
# Funciones físicas del modelo DEE
# ============================================================

def init_cristal_cubico(N, L=1.0, jitter=0.01, seed=42):
    """Inicializa cristal cúbico simple con jitter pequeño en T³."""
    rng = np.random.default_rng(seed)
    n_side = int(np.ceil(N**(1/3)))
    pts = []
    for i in range(n_side):
        for j in range(n_side):
            for k in range(n_side):
                pts.append([i/n_side, j/n_side, k/n_side])
    pts = np.array(pts[:N]) * L
    pts += rng.normal(0, jitter, pts.shape)
    return pts % L

def compute_distances_pbc(points, L):
    """Matriz de distancias con condiciones periódicas."""
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def build_laplacian(points, k_neighbors=30, L=1.0, kernel='inv_d2'):
    """Construye Laplaciano DEE con kernel 1/d² (default) o variantes.
    
    El kernel 1/d² está derivado dimensionalmente: α = d-1 = 2 para 3D.
    Esto NO se modifica.
    """
    N = len(points)
    dist = compute_distances_pbc(points, L)
    np.fill_diagonal(dist, np.inf)
    
    # k vecinos más cercanos
    knn_idx = np.argpartition(dist, k_neighbors, axis=1)[:, :k_neighbors]
    
    W = np.zeros((N, N))
    for i in range(N):
        for j in knn_idx[i]:
            d = dist[i, j]
            if kernel == 'inv_d2':
                w = 1.0 / d**2
            elif kernel == 'inv_d':
                w = 1.0 / d
            elif kernel == 'gauss':
                w = np.exp(-d**2 / 0.05)
            else:
                raise ValueError(f'Kernel {kernel} no reconocido')
            W[i, j] = w
            W[j, i] = w
    
    D = np.diag(W.sum(axis=1))
    return D - W

def laplacian_sqrt_inverse(Lap, eps_reg=1e-3):
    """Devuelve K^(1/2) y K^(-1/2) para cálculo de matriz de covarianza."""
    N = Lap.shape[0]
    L_reg = Lap + eps_reg * np.eye(N)
    eigvals, eigvecs = eigh(L_reg)
    sqrt_K = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
    inv_sqrt_K = eigvecs @ np.diag(1.0 / np.sqrt(eigvals)) @ eigvecs.T
    return sqrt_K, inv_sqrt_K

def E_zp_armonica(Lap):
    """Energía de punto cero del cristal armónico = (1/2) Σ ω_n."""
    eigvals = eigh(Lap, eigvals_only=True)
    eigvals = eigvals[eigvals > 1e-9]
    return 0.5 * np.sqrt(eigvals).sum()

def E_zp_anharmonica(Lap, g4, points, L):
    """Energía de punto cero con corrección anharmónica perturbativa.
    
    Para H = (1/2) φ K φ + (g4/4) φ⁴, el primer orden en perturbación da:
    E_anh = E_arm + (3 g4 / 4) Σ_n |⟨φ²⟩_n|²
    
    Donde ⟨φ²⟩ = (1/2) (K^(-1/2)).
    Para perturbación pequeña, usamos la corrección de Hartree.
    """
    eigvals, eigvecs = eigh(Lap + 1e-3 * np.eye(Lap.shape[0]))
    omegas = np.sqrt(np.maximum(eigvals, 0))
    # Energía armónica
    E_h = 0.5 * omegas.sum()
    # Corrección anharmónica (Hartree): ΔE ~ g4 * Σ_n (1/2ω_n)²
    # phi²_n = 1/(2ω_n), entonces ΔE_n ~ (3g4/4) * φ⁴_n  ~ (3g4/4) * (1/2ω_n)²
    safe_omegas = np.where(omegas > 1e-9, omegas, np.inf)
    correction = (3 * g4 / 16) * np.sum(1.0 / safe_omegas**2)
    return E_h + correction

log("Funciones core definidas.")

## 2. Funciones para análisis de subdominios y entropía

In [ ]:
# ============================================================
# Análisis de subdominios y entropía
# ============================================================

def get_subdomain_mask(points, center, half_size, L=1.0):
    """Máscara booleana de qué nodos están en el cubo [center ± half_size]."""
    inside = np.ones(len(points), dtype=bool)
    for dim in range(3):
        diff_d = points[:, dim] - center[dim]
        diff_d = diff_d - L * np.round(diff_d / L)
        inside &= np.abs(diff_d) < half_size
    return inside

def entropy_subdomain_peschel(mask, sqrt_K, inv_sqrt_K):
    """Entropía de von Neumann del subdominio vía Peschel-Eisert.
    
    Para Hamiltoniano H = (1/2) p² + (1/2) φ K φ con ground-state Gaussiano,
    la entropía del subdominio A es:
    S_A = Σ_k g(ν_k)
    donde ν_k = √(eigvals(X_A · P_A))
    X_A = (1/2) (K^(-1/2))_AA, P_A = (1/2) (K^(1/2))_AA
    g(ν) = (ν+1/2)ln(ν+1/2) - (ν-1/2)ln(ν-1/2)
    """
    idx = np.where(mask)[0]
    if len(idx) < 2:
        return 0.0
    X_A = 0.5 * inv_sqrt_K[np.ix_(idx, idx)]
    P_A = 0.5 * sqrt_K[np.ix_(idx, idx)]
    M = X_A @ P_A
    eig_M = np.real(np.linalg.eigvals(M))
    eig_M = eig_M[eig_M > 1e-12]
    nu = np.sqrt(eig_M)
    nu = np.maximum(nu, 0.5 + 1e-12)
    a = nu + 0.5
    b = nu - 0.5
    s = a * np.log(a) - b * np.log(b)
    return s.sum()

def edges_from_laplacian(Lap, threshold=1e-6):
    """Extrae lista de aristas del Laplaciano."""
    N = Lap.shape[0]
    W = -Lap + np.diag(np.diag(Lap))
    edges = []
    for i in range(N):
        for j in range(i+1, N):
            if abs(W[i, j]) > threshold:
                edges.append((i, j))
    return edges

def n_crosses(mask, edges):
    """Número de aristas que cruzan el borde del subdominio."""
    return sum(1 for (i, j) in edges if mask[i] != mask[j])

def measure_scaling_band(points, sqrt_K, inv_sqrt_K, edges, half_sizes,
                          n_centers, L=1.0, label=""):
    """Mide entropía y aristas-que-cruzan en una banda de escalas.
    
    Devuelve dict con escalas, n_inside promedio, S_A promedio, N_cross promedio.
    """
    rng = np.random.default_rng(42)
    results = {'half_size': [], 'n_inside_mean': [], 'n_inside_std': [],
               'S_A_mean': [], 'S_A_std': [], 'N_cross_mean': [], 'N_cross_std': []}
    
    for hs in half_sizes:
        S_vals, N_vals, In_vals = [], [], []
        for _ in range(n_centers):
            center = rng.random(3)
            mask = get_subdomain_mask(points, center, hs, L)
            n_in = int(mask.sum())
            if n_in < 2:
                continue
            S = entropy_subdomain_peschel(mask, sqrt_K, inv_sqrt_K)
            Nc = n_crosses(mask, edges)
            S_vals.append(S)
            N_vals.append(Nc)
            In_vals.append(n_in)
        
        if S_vals:
            results['half_size'].append(hs)
            results['n_inside_mean'].append(np.mean(In_vals))
            results['n_inside_std'].append(np.std(In_vals))
            results['S_A_mean'].append(np.mean(S_vals))
            results['S_A_std'].append(np.std(S_vals))
            results['N_cross_mean'].append(np.mean(N_vals))
            results['N_cross_std'].append(np.std(N_vals))
    
    return {k: np.array(v) for k, v in results.items()}

def fit_powerlaw(x, y, label=""):
    """Ajuste log-log y = c · x^α. Devuelve α y c."""
    mask = (x > 0) & (y > 0)
    if mask.sum() < 3:
        return np.nan, np.nan, np.nan
    slope, intercept, r_value, _, _ = linregress(np.log(x[mask]), np.log(y[mask]))
    return slope, np.exp(intercept), r_value**2

log("Funciones de análisis listas.")

## 3. MC para generar configuraciones (cristal homogéneo y glassy)

In [ ]:
# ============================================================
# MC para generar configuraciones (annealing como SIM 18/19)
# ============================================================

def funcional_F(points, k=30, L=1.0):
    """Funcional F = N · var(κ_local) — el de SIM 18/19.
    
    Como aproximación rápida (porque κ_Ollivier es caro), uso el proxy:
    κ_local_i = mean degree weight - degree(i)
    Esto preserva la idea de variabilidad de la curvatura local.
    """
    N = len(points)
    dist = compute_distances_pbc(points, L)
    np.fill_diagonal(dist, np.inf)
    knn_idx = np.argpartition(dist, k, axis=1)[:, :k]
    
    # Peso por nodo: suma de 1/d² a sus vecinos
    weight_per_node = np.zeros(N)
    for i in range(N):
        for j in knn_idx[i]:
            weight_per_node[i] += 1.0 / dist[i, j]**2
    
    # κ_local proxy: desviación del peso al promedio
    kappa_local = weight_per_node - weight_per_node.mean()
    return N * np.var(kappa_local)

def annealing_simple(points_init, k=30, L=1.0,
                     temperatures=[43.4, 14.47, 4.34, 0.0],
                     pasos_por_fase=None,
                     sigma_pos=0.01, seed=123):
    """Annealing simulado siguiendo schedule de SIM 19.
    
    Default: 4 fases con T = θ_D, θ_D/3, θ_D/10, 0
    Pasos por fase: ajustados a tamaño chico para Colab.
    """
    if pasos_por_fase is None:
        pasos_por_fase = [3000, 3000, 2000, 4000]  # total 12000 pasos
    
    rng = np.random.default_rng(seed)
    points = points_init.copy()
    N = len(points)
    F_curr = funcional_F(points, k, L)
    history = {'T': [], 'step': [], 'F': []}
    
    step_global = 0
    for T, n_steps in zip(temperatures, pasos_por_fase):
        for _ in range(n_steps):
            i = rng.integers(N)
            old_pos = points[i].copy()
            points[i] = (points[i] + sigma_pos * rng.standard_normal(3)) % L
            F_new = funcional_F(points, k, L)
            dF = F_new - F_curr
            
            if T > 0:
                accept = dF < 0 or rng.random() < np.exp(-dF / T)
            else:
                accept = dF < 0
            
            if accept:
                F_curr = F_new
            else:
                points[i] = old_pos
            
            step_global += 1
        
        history['T'].append(T)
        history['step'].append(step_global)
        history['F'].append(F_curr)
    
    return points, history

log("MC funcional listo.")

## 4. Test 1+2: Escalado en bandas mesoscópicas con sweep de N

**El test crítico**: separamos las escalas en tres bandas (UV, mesoscópica, IR) y medimos el escalado en cada una. Si el régimen mesoscópico tiene exponente distinto del UV, eso confirma que los tests anteriores estaban en el régimen equivocado.

In [ ]:
# ============================================================
# TEST 1+2: Escalado en bandas + sweep de N
# ============================================================

log("\n" + "="*70)
log("TEST 1+2: Escalado en bandas mesoscópicas + sweep de N")
log("="*70)

# Sweep de tamaños — siguiendo SIM 9d (que llegó a N=10648 = 22³)
# En Colab Pro con CPU normal: N=8000 toma ~30 min para todo el pipeline
N_values = [1000, 2000, 4000, 8000]

# Bandas de escala (en unidades de la caja L=1):
# - UV: subdominios chicos (pocos nodos por subdominio)
# - Mesoscópica: subdominios donde N_inside ≥ 50 (régimen donde aplica EFT)
# - IR: subdominios grandes pero menores que L/2
#
# Para N grande, las bandas se redefinen para mantener N_inside en el rango correcto.

results_per_N = {}

for N in N_values:
    log(f"\n--- Procesando N = {N} ---")
    t_start = time.time()
    
    # Inicializar cristal
    points = init_cristal_cubico(N, L=1.0, jitter=0.01, seed=42)
    
    # Construir Laplaciano y K^(±1/2)
    log(f"  Construyendo Laplaciano (N={N}, k=30)...")
    Lap = build_laplacian(points, k_neighbors=30, L=1.0)
    log(f"    Tiempo: {time.time() - t_start:.1f}s")
    
    log("  Calculando K^(1/2) y K^(-1/2)...")
    t1 = time.time()
    sqrt_K, inv_sqrt_K = laplacian_sqrt_inverse(Lap)
    log(f"    Tiempo: {time.time() - t1:.1f}s")
    
    log("  Extrayendo aristas...")
    edges = edges_from_laplacian(Lap)
    log(f"    Aristas totales: {len(edges)}")
    
    # Bandas adaptadas a N. Régimen mesoscópico = donde N_inside ≥ 50 (suficiente coarse-graining)
    # pero ℓ << L/2 (no satura la caja).
    # Para N=8000, a≈0.05; meso ℓ ~ 3a-10a = 0.15-0.5, hay que limitar a 0.40.
    # Para N=1000, a≈0.10; meso ℓ ~ 3a-6a = 0.30-0.60 -> nos quedamos con 0.30-0.40.
    a = N**(-1/3)
    bands = {
        'UV':   np.array([1.0*a, 1.3*a, 1.6*a, 2.0*a, 2.5*a]),
        'meso': np.array([3*a, 3.5*a, 4*a, 4.5*a, 5*a, 6*a, 7*a, 8*a]),
        'IR':   np.array([0.20, 0.225, 0.25, 0.275, 0.30, 0.325, 0.35]),
    }
    
    # Filtrar bandas: half_size debe ser < 0.40 (subdominio < L/2.5)
    # Pero garantizar que cada banda tenga al menos 4 puntos para ajuste log-log fiable
    L_max = 0.40  # half_size máximo permitido
    L_min = 0.025  # half_size mínimo (evita subdominios vacíos)
    for k_band in bands:
        bands[k_band] = bands[k_band][(bands[k_band] < L_max) & (bands[k_band] > L_min)]
        # Si quedan pocos puntos, expandir la banda
        if len(bands[k_band]) < 4:
            if k_band == 'meso':
                # Generar 4-6 puntos en log-spacing entre el spacing y L_max/2
                bands[k_band] = np.geomspace(max(2.5*a, L_min), min(0.30, L_max), 5)
            elif k_band == 'UV':
                bands[k_band] = np.geomspace(max(1.0*a, L_min), max(2.5*a, L_min*1.5), 5)
            elif k_band == 'IR':
                bands[k_band] = np.linspace(0.20, 0.35, 5)
    
    # Verificación: si una banda tiene < 3 puntos, marcar como insuficiente
    log(f"  Bandas con N={N} (a={a:.4f}):")
    for k_band, hs in bands.items():
        log(f"    {k_band}: ℓ ∈ [{hs.min():.3f}, {hs.max():.3f}] ({len(hs)} pts)")
    
    log("  Midiendo escalado en bandas...")
    n_centers = max(30, min(150, 200000 // N))  # más centros para N chico
    
    band_results = {}
    for band_name, half_sizes in bands.items():
        if len(half_sizes) < 3:
            continue
        log(f"    Banda {band_name}: ℓ ∈ [{half_sizes.min():.4f}, {half_sizes.max():.4f}], n_centers={n_centers}")
        t_band = time.time()
        res = measure_scaling_band(points, sqrt_K, inv_sqrt_K, edges,
                                    half_sizes, n_centers, L=1.0, label=band_name)
        band_results[band_name] = res
        
        # Ajuste rápido
        if len(res['half_size']) >= 3:
            d_S, _, R2_S = fit_powerlaw(res['half_size'], res['S_A_mean'])
            d_E, _, R2_E = fit_powerlaw(res['half_size'], res['N_cross_mean'])
            log(f"      S ∝ ℓ^{d_S:.3f} (R²={R2_S:.4f}), N_cross ∝ ℓ^{d_E:.3f} (R²={R2_E:.4f})")
        log(f"      Tiempo banda: {time.time() - t_band:.1f}s")
    
    results_per_N[N] = {
        'bands': band_results,
        'n_edges_total': len(edges),
        'a_spacing': a,
    }
    
    log(f"  Total para N={N}: {time.time() - t_start:.1f}s")
    
    # CHECKPOINT: guardamos después de cada N por si se cae la sesión
    with open(f'{OUTPUT_DIR}/datos/checkpoint_N{N}.pkl', 'wb') as f:
        pickle.dump(results_per_N, f)
    log(f"  Checkpoint guardado: checkpoint_N{N}.pkl")
    
    # Liberar memoria
    del Lap, sqrt_K, inv_sqrt_K, edges
    import gc
    gc.collect()

# Guardar resultados intermedios
with open(f'{OUTPUT_DIR}/datos/test_1_2_scaling.pkl', 'wb') as f:
    pickle.dump(results_per_N, f)

log("\n=== Test 1+2 completado ===")

## 5. Análisis: ¿hay diferencias entre bandas?

In [ ]:
# ============================================================
# Análisis comparativo entre bandas y N
# ============================================================

log("\n" + "="*70)
log("ANÁLISIS: exponentes por banda y por N")
log("="*70)

log(f"\n{'N':<8} {'banda':<8} {'d_S':<10} {'d_E (cross)':<12} {'R²_S':<8} {'R²_E':<8}")
log("-" * 60)

exponents = {}
for N, results in results_per_N.items():
    exponents[N] = {}
    for band_name, band_data in results['bands'].items():
        if len(band_data.get('half_size', [])) >= 3:
            d_S, _, R2_S = fit_powerlaw(band_data['half_size'], band_data['S_A_mean'])
            d_E, _, R2_E = fit_powerlaw(band_data['half_size'], band_data['N_cross_mean'])
            exponents[N][band_name] = {'d_S': d_S, 'd_E': d_E, 'R2_S': R2_S, 'R2_E': R2_E}
            log(f"{N:<8} {band_name:<8} {d_S:<10.3f} {d_E:<12.3f} {R2_S:<8.4f} {R2_E:<8.4f}")

# Test de extrapolación al continuo: para cada banda, ajustar d(N) → d(∞)
log("\n=== Extrapolación N → ∞ ===")
log("d_S(N) = d_∞ + a/N^(1/3)  (corrección estándar de tamaño finito)")

for band_name in ['UV', 'meso', 'IR']:
    Ns_avail = [N for N in N_values if band_name in exponents.get(N, {})]
    if len(Ns_avail) < 3:
        continue
    
    log(f"\nBanda {band_name}:")
    Ns_arr = np.array(Ns_avail)
    d_S_arr = np.array([exponents[N][band_name]['d_S'] for N in Ns_avail])
    d_E_arr = np.array([exponents[N][band_name]['d_E'] for N in Ns_avail])
    
    # Ajuste lineal en 1/N^(1/3)
    x_fit = 1.0 / Ns_arr**(1/3)
    
    if len(d_S_arr) >= 2:
        slope_S, intercept_S, _, _, _ = linregress(x_fit, d_S_arr)
        d_S_inf = intercept_S
        log(f"  d_S por N: {dict(zip(Ns_avail, d_S_arr.round(3)))}")
        log(f"  d_S(∞) = {d_S_inf:.3f} (extrapolación)")
        
        slope_E, intercept_E, _, _, _ = linregress(x_fit, d_E_arr)
        d_E_inf = intercept_E
        log(f"  d_E por N: {dict(zip(Ns_avail, d_E_arr.round(3)))}")
        log(f"  d_E(∞) = {d_E_inf:.3f} (extrapolación)")
        
        # Predicción de w cosmológico
        # Si ρ_E = E/V con E ∝ S y S ∝ V^(d_S/3), entonces ρ_E ∝ V^(d_S/3 - 1)
        # ρ ∝ a^(-3(1+w)) → 3(1+w) = -3·(d_S/3 - 1) = 3 - d_S
        # → w = -d_S/3
        w_predicted = -d_S_inf / 3
        log(f"  Predicción w cosmológico (banda {band_name}): w = {w_predicted:+.3f}")
        
        if band_name == 'meso':
            log(f"  ▶ ESTE ES EL VALOR FÍSICAMENTE RELEVANTE (régimen mesoscópico)")

# Guardar exponentes
with open(f'{OUTPUT_DIR}/datos/exponentes_por_banda.json', 'w') as f:
    json.dump({str(N): {b: {k: float(v) for k, v in d.items()} for b, d in e.items()} 
               for N, e in exponents.items()}, f, indent=2)

log("\n=== Análisis completado ===")

## 6. Test 3: w cosmológico para tres regímenes

In [ ]:
# ============================================================
# TEST 3: w cosmológico para tres regímenes:
# (a) cristal homogéneo (control: w ≈ +0.07)
# (b) configuración glassy (annealing simulado)
# (c) cristal con anharmonicidad φ⁴
# ============================================================

log("\n" + "="*70)
log("TEST 3: w cosmológico para tres regímenes")
log("="*70)

def compute_w_armonico(points, c_s=2.186, k=30):
    """Calcula w siguiendo SIM 16/17.
    
    Variar L: [-2%, -1%, 0, +1%, +2%]. Para cada escala calcular E_zp.
    Extraer w = -P/(ρ·c_s²) con P = -dE/dV.
    """
    epsilons = [-0.02, -0.01, 0.0, 0.01, 0.02]
    Es, Vs = [], []
    for eps in epsilons:
        L_scale = 1.0 + eps
        Lap = build_laplacian(points * L_scale, k_neighbors=k, L=L_scale)
        E = E_zp_armonica(Lap)
        Es.append(E)
        Vs.append(L_scale**3)
    Es, Vs = np.array(Es), np.array(Vs)
    idx_0 = 2
    dE_dV = (Es[idx_0+1] - Es[idx_0-1]) / (Vs[idx_0+1] - Vs[idx_0-1])
    P = -dE_dV
    rho = Es[idx_0] / Vs[idx_0]
    return P / (rho * c_s**2), Es[idx_0]

def compute_w_anharmonico(points, g4, c_s=2.186, k=30):
    """Calcula w incluyendo término anharmónico g4·φ⁴."""
    epsilons = [-0.02, -0.01, 0.0, 0.01, 0.02]
    Es, Vs = [], []
    for eps in epsilons:
        L_scale = 1.0 + eps
        Lap = build_laplacian(points * L_scale, k_neighbors=k, L=L_scale)
        E = E_zp_anharmonica(Lap, g4, points * L_scale, L_scale)
        Es.append(E)
        Vs.append(L_scale**3)
    Es, Vs = np.array(Es), np.array(Vs)
    idx_0 = 2
    dE_dV = (Es[idx_0+1] - Es[idx_0-1]) / (Vs[idx_0+1] - Vs[idx_0-1])
    P = -dE_dV
    rho = Es[idx_0] / Vs[idx_0]
    return P / (rho * c_s**2), Es[idx_0]

# === Caso (a): cristal homogéneo ===
log("\n--- (a) Cristal homogéneo (control) ---")
N_test = 4000
points_homo = init_cristal_cubico(N_test, jitter=0.01, seed=42)
w_homo, E_homo = compute_w_armonico(points_homo)
log(f"N = {N_test}, w_homogéneo = {w_homo:+.5f}, E_zp = {E_homo:.2f}")
log(f"  Esperado de SIM 16/17: w ≈ +0.0698")
log(f"  Discrepancia: {abs(w_homo - 0.0698)/0.0698*100:.2f}%")

# === Caso (b): configuración glassy ===
log("\n--- (b) Configuración glassy (annealing) ---")
log("Ejecutando annealing simulado...")
t1 = time.time()
points_glassy, hist = annealing_simple(points_homo.copy(), 
                                        pasos_por_fase=[2000, 2000, 1500, 3000])
log(f"  Annealing: {time.time() - t1:.1f}s")
log(f"  F_final = {hist['F'][-1]:.3f}")
w_glassy, E_glassy = compute_w_armonico(points_glassy)
log(f"w_glassy = {w_glassy:+.5f}, E_zp = {E_glassy:.2f}")
log(f"  Diferencia con homogéneo: {(w_glassy - w_homo)*100:.4f}% (en valor absoluto del exponente)")

# === Caso (c): anharmonicidad ===
log("\n--- (c) Cristal con anharmonicidad φ⁴ ---")
# g4 más altos para ver efecto: con perturbación chica el cambio es < 0.1%
g4_values = [0.0, 0.1, 1.0, 10.0, 100.0]
log(f"Probando g4 ∈ {g4_values}")
log("(Valores altos: el cálculo es perturbativo Hartree, válido si ΔE < E_armónico)")

ws_anh = []
for g4 in g4_values:
    w_anh, E_anh = compute_w_anharmonico(points_homo, g4)
    ws_anh.append(w_anh)
    log(f"  g4 = {g4:6.3f}: w = {w_anh:+.5f}, E = {E_anh:.2f}")

log("\nObservación clave:")
log(f"  w varía con g4? {abs(max(ws_anh) - min(ws_anh)):.5f} (rango)")
log(f"  Si la anharmonicidad realmente cambia w, esto es la dirección hacia Λ")

# === Caso (d): glassy + anharmonicidad ===
log("\n--- (d) Glassy + anharmonicidad (combinación) ---")
log("Tu hipótesis combinatoria: ¿juntos dan algo distinto que cada uno?")
ws_combo = []
for g4 in g4_values:
    w_combo, _ = compute_w_anharmonico(points_glassy, g4)
    ws_combo.append(w_combo)
    log(f"  glassy + g4={g4:6.3f}: w = {w_combo:+.5f}")

# Test de combinatoria: si las contribuciones fueran aditivas:
# w_combo(g4) ≈ w_glassy + (w_anh(g4) - w_homo)
log("\nTest de aditividad (combinatoria lineal):")
for i, g4 in enumerate(g4_values):
    if i == 0:
        continue
    additive = w_glassy + (ws_anh[i] - w_homo)
    actual = ws_combo[i]
    log(f"  g4={g4:6.3f}: aditivo={additive:+.5f}, real={actual:+.5f}, diff={actual-additive:+.5f}")
    if abs(actual - additive) > 0.001:
        log(f"      → NO aditivo, hay efecto interactivo")

# Guardar resultados
test3_results = {
    'w_homogeneo': float(w_homo),
    'w_glassy': float(w_glassy),
    'g4_values': g4_values,
    'ws_anharmonicos': [float(w) for w in ws_anh],
    'ws_glassy_anharmonicos': [float(w) for w in ws_combo],
    'F_glassy_final': float(hist['F'][-1]),
}
with open(f'{OUTPUT_DIR}/datos/test_3_w_cosmologico.json', 'w') as f:
    json.dump(test3_results, f, indent=2)

log("\n=== Test 3 completado ===")

## 7. Test 4: Convergencia al continuo de los exponentes

In [ ]:
# ============================================================
# TEST 4: Análisis de convergencia + tests de control
# ============================================================

log("\n" + "="*70)
log("TEST 4: Convergencia al continuo + controles")
log("="*70)

# Control 1: cristal cúbico ideal (sin jitter)
log("\n--- Control 1: cristal ideal sin jitter ---")
for N in [1000, 4000]:
    points_ideal = init_cristal_cubico(N, jitter=0.0, seed=0)
    Lap_ideal = build_laplacian(points_ideal, k_neighbors=30)
    sqrt_K_i, inv_sqrt_K_i = laplacian_sqrt_inverse(Lap_ideal)
    edges_ideal = edges_from_laplacian(Lap_ideal)
    
    a = N**(-1/3)
    half_sizes_meso = np.array([3*a, 4*a, 5*a, 6*a, 8*a, 10*a])
    half_sizes_meso = half_sizes_meso[half_sizes_meso < 0.4]
    
    res = measure_scaling_band(points_ideal, sqrt_K_i, inv_sqrt_K_i, edges_ideal,
                                half_sizes_meso, n_centers=50, L=1.0)
    if len(res['half_size']) >= 3:
        d_S, _, R2 = fit_powerlaw(res['half_size'], res['S_A_mean'])
        log(f"  N={N}: d_S(meso) ideal = {d_S:.3f} (R²={R2:.3f})")

# Control 2: configuración aleatoria (esperamos d_S → 3, ley de volumen)
log("\n--- Control 2: posiciones aleatorias (no cristal) ---")
rng_ctrl = np.random.default_rng(7)
for N in [1000, 4000]:
    points_rand = rng_ctrl.uniform(0, 1, (N, 3))
    Lap_rand = build_laplacian(points_rand, k_neighbors=30)
    sqrt_K_r, inv_sqrt_K_r = laplacian_sqrt_inverse(Lap_rand)
    edges_rand = edges_from_laplacian(Lap_rand)
    
    a = N**(-1/3)
    half_sizes_meso = np.array([3*a, 4*a, 5*a, 6*a, 8*a, 10*a])
    half_sizes_meso = half_sizes_meso[half_sizes_meso < 0.4]
    
    res = measure_scaling_band(points_rand, sqrt_K_r, inv_sqrt_K_r, edges_rand,
                                half_sizes_meso, n_centers=50, L=1.0)
    if len(res['half_size']) >= 3:
        d_S, _, R2 = fit_powerlaw(res['half_size'], res['S_A_mean'])
        log(f"  N={N}: d_S(meso) random = {d_S:.3f} (R²={R2:.3f})")

# Tabla resumen final
log("\n" + "="*70)
log("RESUMEN GLOBAL DEL EXPERIMENTO")
log("="*70)

log("\nExponente d_S por banda y N:")
log(f"{'N':>6} | {'UV':>8} {'meso':>8} {'IR':>8}")
log("-" * 35)
for N in N_values:
    if N in exponents:
        row = f"{N:>6} |"
        for band in ['UV', 'meso', 'IR']:
            if band in exponents[N]:
                row += f" {exponents[N][band]['d_S']:>8.3f}"
            else:
                row += f" {'—':>8}"
        log(row)

log("\nPredicciones de w cosmológico:")
for band in ['UV', 'meso', 'IR']:
    Ns_avail = [N for N in N_values if band in exponents.get(N, {})]
    if len(Ns_avail) >= 2:
        d_S_arr = np.array([exponents[N][band]['d_S'] for N in Ns_avail])
        x_fit = 1.0 / np.array(Ns_avail)**(1/3)
        slope, intercept, _, _, _ = linregress(x_fit, d_S_arr)
        d_S_inf = intercept
        w_pred = -d_S_inf / 3
        log(f"  Banda {band}: d_S(∞) = {d_S_inf:+.3f} → w = {w_pred:+.3f}")

log("\n=== Test 4 completado ===")

## 7b. Test 5: ¿La anharmonicidad modifica el ESCALADO de S(ℓ)?

Test 3 mide w pero solo via E_zp. **Este test mide si la anharmonicidad cambia el régimen holográfico** de la entropía. Usamos Hartree autoconsistente: $K_\text{eff} = K + 3 g_4 \langle \phi^2 \rangle$, después aplicamos Peschel-Eisert con $K_\text{eff}$. Si $d_S$(meso) cambia con $g_4$, la anharmonicidad sí modifica la holografía.

In [ ]:
# ============================================================
# TEST 5: Escalado anharmónico de S(ℓ) via Hartree autoconsistente
# ============================================================

log("\n" + "="*70)
log("TEST 5: Escalado de entropía con anharmonicidad (Hartree)")
log("="*70)

def hartree_self_consistent(Lap, g4, max_iter=20, tol=1e-4, eps_reg=1e-3):
    """Calcula Laplaciano efectivo Hartree: K_eff = K + 3 g4 * diag(<phi²>)
    
    <phi²>_i = (1/2) (K_eff^(-1/2))_ii  (autoconsistente)
    Iterar hasta convergencia.
    
    Devuelve (K_eff, sqrt_K_eff, inv_sqrt_K_eff, converged).
    """
    N = Lap.shape[0]
    K_eff = Lap.copy()
    converged = False
    
    for it in range(max_iter):
        K_reg = K_eff + eps_reg * np.eye(N)
        eigvals, eigvecs = eigh(K_reg)
        # Inverse square root
        safe_eigvals = np.maximum(eigvals, 1e-9)
        inv_sqrt = eigvecs @ np.diag(1.0 / np.sqrt(safe_eigvals)) @ eigvecs.T
        
        # phi²_i = (1/2) (K^(-1/2))_ii
        phi_sq = 0.5 * np.diag(inv_sqrt)
        
        # Nueva corrección Hartree
        K_new = Lap + np.diag(3 * g4 * phi_sq)
        
        # Test de convergencia
        diff = np.linalg.norm(K_new - K_eff) / (np.linalg.norm(K_eff) + 1e-12)
        K_eff = K_new
        
        if diff < tol:
            converged = True
            break
    
    K_reg = K_eff + eps_reg * np.eye(N)
    eigvals, eigvecs = eigh(K_reg)
    safe_eigvals = np.maximum(eigvals, 1e-9)
    sqrt_K_eff = eigvecs @ np.diag(np.sqrt(safe_eigvals)) @ eigvecs.T
    inv_sqrt_K_eff = eigvecs @ np.diag(1.0 / np.sqrt(safe_eigvals)) @ eigvecs.T
    
    return K_eff, sqrt_K_eff, inv_sqrt_K_eff, converged, it+1

log("\nProbando S(ℓ) con corrección Hartree para varios g4...")

# Usar N=2000 (compromiso velocidad/precisión para este test específico)
N_test5 = 2000
log(f"N = {N_test5} (fijo para este test)")

points_t5 = init_cristal_cubico(N_test5, jitter=0.01, seed=42)
Lap_t5 = build_laplacian(points_t5, k_neighbors=30)
edges_t5 = edges_from_laplacian(Lap_t5)

a_t5 = N_test5**(-1/3)
# Banda mesoscópica garantizando 4-6 puntos
half_sizes_meso_t5 = np.geomspace(max(2.5*a_t5, 0.025), min(0.30, 0.40), 6)
log(f"Banda mesoscópica: ℓ ∈ [{half_sizes_meso_t5.min():.4f}, {half_sizes_meso_t5.max():.4f}] ({len(half_sizes_meso_t5)} pts)")

g4_test5 = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0]
test5_results = {}

for g4 in g4_test5:
    log(f"\n  g4 = {g4}:")
    t1 = time.time()
    K_eff, sqrt_K_eff, inv_sqrt_K_eff, conv, n_iter = hartree_self_consistent(Lap_t5, g4)
    log(f"    Hartree: convergencia={conv} en {n_iter} iter ({time.time()-t1:.1f}s)")
    
    # Medir S por bandas con K_eff
    res = measure_scaling_band(points_t5, sqrt_K_eff, inv_sqrt_K_eff, edges_t5,
                                half_sizes_meso_t5, n_centers=50, L=1.0)
    if len(res['half_size']) >= 3:
        d_S, _, R2 = fit_powerlaw(res['half_size'], res['S_A_mean'])
        log(f"    d_S(meso) = {d_S:.3f} (R²={R2:.3f})")
        log(f"    S(ℓ_min)={res['S_A_mean'][0]:.3f}, S(ℓ_max)={res['S_A_mean'][-1]:.3f}")
        test5_results[g4] = {
            'd_S': float(d_S),
            'R2': float(R2),
            'half_sizes': res['half_size'].tolist(),
            'S_A_mean': res['S_A_mean'].tolist(),
            'converged': conv,
            'n_iter': n_iter,
        }
    else:
        log(f"    Datos insuficientes")
        test5_results[g4] = {'error': 'insufficient_data'}

log("\n=== Resumen Test 5: ¿d_S cambia con g4? ===")
log(f"{'g4':<10} {'d_S':<10} {'cambio vs g4=0':<15}")
log("-" * 35)
d_S_0 = test5_results.get(0.0, {}).get('d_S', None)
for g4 in g4_test5:
    if 'd_S' in test5_results.get(g4, {}):
        d_S = test5_results[g4]['d_S']
        delta = d_S - d_S_0 if d_S_0 is not None else None
        log(f"{g4:<10.4f} {d_S:<10.3f} {delta:+.3f}" if delta is not None else f"{g4:<10.4f} {d_S:<10.3f}")

with open(f'{OUTPUT_DIR}/datos/test5_anharm_scaling.json', 'w') as f:
    json.dump({str(k): v for k, v in test5_results.items()}, f, indent=2)

log("\n=== Test 5 completado ===")

## 7c. Test 6: Régimen no perturbativo de g4 — ¿hasta dónde Hartree es válido?

In [ ]:
# ============================================================
# TEST 6: Hasta dónde llega el régimen no perturbativo
# ============================================================

log("\n" + "="*70)
log("TEST 6: Régimen no perturbativo de g4")
log("="*70)

log("\nEvaluamos cuándo la corrección Hartree deja de converger / E_anh > E_arm")

g4_extended = [0.0, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]
test6_results = []

Lap_ref = Lap_t5  # mismo Laplaciano que Test 5
E_arm_ref = E_zp_armonica(Lap_ref)
log(f"E_armónico de referencia (g4=0): {E_arm_ref:.2f}")
log("")

for g4 in g4_extended:
    t1 = time.time()
    try:
        K_eff, _, _, conv, n_iter = hartree_self_consistent(Lap_ref, g4, max_iter=30)
        E_arm_eff = E_zp_armonica(K_eff)  # E del Laplaciano efectivo
        
        # E total con corrección Hartree:  E_h(K_eff) - (3 g4 / 4) Σ phi²_i²
        # (la última parte resta el doble conteo de la interacción)
        eigvals, eigvecs = eigh(K_eff + 1e-3 * np.eye(K_eff.shape[0]))
        safe = np.maximum(eigvals, 1e-9)
        inv_sqrt = eigvecs @ np.diag(1.0/np.sqrt(safe)) @ eigvecs.T
        phi_sq = 0.5 * np.diag(inv_sqrt)
        E_double_count = (3 * g4 / 4) * np.sum(phi_sq**2)
        E_total = E_arm_eff - E_double_count
        
        ratio = E_total / E_arm_ref if E_arm_ref > 0 else np.nan
        valid = conv and ratio > 0 and ratio < 10
        
        log(f"g4={g4:>8.2f}: conv={conv} ({n_iter} iter), E_total/E_arm={ratio:.3f}, válido={valid} [{time.time()-t1:.1f}s]")
        
        test6_results.append({
            'g4': g4,
            'converged': conv,
            'n_iter': n_iter,
            'E_total': float(E_total),
            'E_arm_ref': float(E_arm_ref),
            'ratio': float(ratio),
            'valid': valid,
        })
    except Exception as e:
        log(f"g4={g4:>8.2f}: ERROR - {str(e)[:60]}")
        test6_results.append({'g4': g4, 'error': str(e)})

# Encontrar el último g4 válido
valid_g4s = [r['g4'] for r in test6_results if r.get('valid', False)]
if valid_g4s:
    log(f"\n>> Régimen perturbativo válido hasta g4 ≈ {max(valid_g4s)}")
    log(f">> Más allá de eso, se necesitan métodos no-Gaussianos (DMRG, QMC, VMC)")

with open(f'{OUTPUT_DIR}/datos/test6_nonperturbative.json', 'w') as f:
    json.dump(test6_results, f, indent=2)

log("\n=== Test 6 completado ===")

## 7d. Test 7: Espectro vibracional armónico vs anharmónico

In [ ]:
# ============================================================
# TEST 7: Espectro de modos vibracionales armónico vs anharmónico
# ============================================================

log("\n" + "="*70)
log("TEST 7: Espectro vibracional con/sin anharmonicidad")
log("="*70)

log("\nComparamos espectros para detectar:")
log("  - apertura de gap en IR (haría w más negativo)")
log("  - desplazamiento global del espectro (cambia E_zp)")
log("  - cambios de densidad espectral (cambia dimensión espectral)")

# Espectro armónico (g4 = 0)
eigvals_arm = eigh(Lap_t5 + 1e-3 * np.eye(Lap_t5.shape[0]), eigvals_only=True)
omegas_arm = np.sqrt(np.maximum(eigvals_arm, 0))

test7_results = {'armonico': {
    'omega_min': float(omegas_arm[1]),  # excluyendo modo cero
    'omega_max': float(omegas_arm[-1]),
    'omega_mean': float(omegas_arm.mean()),
    'omega_median': float(np.median(omegas_arm)),
    'E_zp': float(0.5 * omegas_arm[1:].sum()),
    'gap_IR': float(omegas_arm[1] - omegas_arm[0]),  # gap entre modo cero y primero
}}

log(f"\nEspectro armónico:")
log(f"  ω_min (1er modo no-cero):    {omegas_arm[1]:.5f}")
log(f"  ω_max:                        {omegas_arm[-1]:.4f}")
log(f"  ⟨ω⟩:                          {omegas_arm.mean():.4f}")
log(f"  Gap IR (ω_1 - ω_0):           {omegas_arm[1] - omegas_arm[0]:.5f}")

# Espectro anharmónico (Hartree con varios g4)
g4_test7 = [0.1, 1.0, 10.0]
test7_results['anharmonico'] = {}

for g4 in g4_test7:
    K_eff, _, _, conv, _ = hartree_self_consistent(Lap_t5, g4)
    eigvals_anh = eigh(K_eff + 1e-3 * np.eye(K_eff.shape[0]), eigvals_only=True)
    omegas_anh = np.sqrt(np.maximum(eigvals_anh, 0))
    
    log(f"\nEspectro anharmónico (g4={g4}, conv={conv}):")
    log(f"  ω_min (1er modo no-cero):    {omegas_anh[1]:.5f} (Δ vs arm: {omegas_anh[1]-omegas_arm[1]:+.5f})")
    log(f"  ω_max:                        {omegas_anh[-1]:.4f} (Δ: {omegas_anh[-1]-omegas_arm[-1]:+.4f})")
    log(f"  ⟨ω⟩:                          {omegas_anh.mean():.4f} (Δ: {omegas_anh.mean()-omegas_arm.mean():+.4f})")
    log(f"  Gap IR (ω_1 - ω_0):           {omegas_anh[1] - omegas_anh[0]:.5f}")
    
    test7_results['anharmonico'][str(g4)] = {
        'omega_min': float(omegas_anh[1]),
        'omega_max': float(omegas_anh[-1]),
        'omega_mean': float(omegas_anh.mean()),
        'omega_median': float(np.median(omegas_anh)),
        'E_zp': float(0.5 * omegas_anh[1:].sum()),
        'gap_IR': float(omegas_anh[1] - omegas_anh[0]),
        'omegas_full': omegas_anh.tolist(),
    }

test7_results['armonico']['omegas_full'] = omegas_arm.tolist()

# Análisis: ¿se abre gap o se desplaza el espectro?
log("\n=== Diagnóstico Test 7 ===")
for g4 in g4_test7:
    anh = test7_results['anharmonico'][str(g4)]
    delta_omega_min = anh['omega_min'] - test7_results['armonico']['omega_min']
    delta_E = anh['E_zp'] - test7_results['armonico']['E_zp']
    log(f"\ng4 = {g4}:")
    if abs(delta_omega_min) > 0.01:
        log(f"  → Modificación significativa del modo IR (Δω_min = {delta_omega_min:+.4f})")
        log(f"    Esto cambia el comportamiento de larga distancia y puede dar w distinto")
    else:
        log(f"  → IR no modificado significativamente (Δω_min = {delta_omega_min:+.5f})")
    log(f"  → ΔE_zp = {delta_E:+.2f} ({delta_E/test7_results['armonico']['E_zp']*100:+.2f}%)")

with open(f'{OUTPUT_DIR}/datos/test7_espectro.json', 'w') as f:
    # Guardar sin los arrays completos para JSON ligero
    summary = {
        'armonico': {k: v for k, v in test7_results['armonico'].items() if k != 'omegas_full'},
        'anharmonico': {g: {k: v for k, v in d.items() if k != 'omegas_full'} 
                          for g, d in test7_results['anharmonico'].items()}
    }
    json.dump(summary, f, indent=2)

# Pickle con espectros completos para post-análisis
with open(f'{OUTPUT_DIR}/datos/test7_espectro_completo.pkl', 'wb') as f:
    pickle.dump(test7_results, f)

log("\n=== Test 7 completado ===")

## 8. Generación de figuras

In [ ]:
# ============================================================
# Figuras
# ============================================================

log("\nGenerando figuras...")

# Figura 1: S_A vs ℓ por banda y N
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, band in zip(axes, ['UV', 'meso', 'IR']):
    for N in N_values:
        if N in results_per_N and band in results_per_N[N]['bands']:
            data = results_per_N[N]['bands'][band]
            if len(data.get('half_size', [])) >= 3:
                ax.errorbar(data['half_size'], data['S_A_mean'],
                           yerr=data['S_A_std']/np.sqrt(50),
                           label=f'N={N}', marker='o')
    ax.set_xlabel('ℓ (half-size)')
    ax.set_ylabel('S_A (entanglement entropy)')
    ax.set_title(f'Banda {band}')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig1_entropia_por_banda.png', dpi=150, bbox_inches='tight')
plt.close()
log("  fig1: entropía por banda — guardada")

# Figura 2: extrapolación N → ∞
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
for band in ['UV', 'meso', 'IR']:
    Ns_avail = [N for N in N_values if band in exponents.get(N, {})]
    if len(Ns_avail) >= 2:
        d_S_arr = np.array([exponents[N][band]['d_S'] for N in Ns_avail])
        x_fit = 1.0 / np.array(Ns_avail)**(1/3)
        ax.plot(x_fit, d_S_arr, 'o-', label=band, markersize=10)
        # Extrapolación
        slope, intercept, _, _, _ = linregress(x_fit, d_S_arr)
        x_ext = np.linspace(0, x_fit.max() * 1.2, 50)
        ax.plot(x_ext, slope*x_ext + intercept, '--', alpha=0.5)
        ax.scatter([0], [intercept], marker='*', s=200, zorder=5)
ax.axhline(2, color='gray', linestyle=':', label='ley de área (d=2)')
ax.axhline(3, color='gray', linestyle='--', label='ley de volumen (d=3)')
ax.axhline(1, color='gray', linestyle='-.', label='ley lineal (d=1)')
ax.set_xlabel('1 / N^(1/3)')
ax.set_ylabel('d_S (exponente)')
ax.set_title('Extrapolación al continuo: d_S vs N')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig2_extrapolacion_continuo.png', dpi=150, bbox_inches='tight')
plt.close()
log("  fig2: extrapolación al continuo — guardada")

# Figura 3: w cosmológico para tres regímenes
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.plot(g4_values, ws_anh, 'o-', label='homogéneo + g4·φ⁴', linewidth=2, markersize=8)
ax.plot(g4_values, ws_combo, 's-', label='glassy + g4·φ⁴', linewidth=2, markersize=8)
ax.axhline(w_homo, color='blue', linestyle=':', label=f'homogéneo (w={w_homo:+.3f})')
ax.axhline(w_glassy, color='orange', linestyle=':', label=f'glassy puro (w={w_glassy:+.3f})')
ax.axhline(-1, color='red', linestyle='--', label='Λ exacta (w=-1)')
ax.axhline(-2/3, color='purple', linestyle='--', label='holográfico (w=-2/3)')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('g4 (anharmonicidad)')
ax.set_ylabel('w cosmológico')
ax.set_title('w como función de anharmonicidad y régimen glassy')
ax.set_xscale('symlog', linthresh=0.001)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig3_w_cosmologico.png', dpi=150, bbox_inches='tight')
plt.close()
log("  fig3: w cosmológico — guardada")

# Figura 4: aristas que cruzan vs entropía
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, band in zip(axes, ['UV', 'meso', 'IR']):
    for N in N_values:
        if N in results_per_N and band in results_per_N[N]['bands']:
            data = results_per_N[N]['bands'][band]
            if len(data.get('half_size', [])) >= 3:
                ax.plot(data['N_cross_mean'], data['S_A_mean'], 'o-',
                       label=f'N={N}', markersize=8)
    ax.set_xlabel('N_aristas que cruzan')
    ax.set_ylabel('S_A (entropía)')
    ax.set_title(f'Banda {band}: relación S vs aristas')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig4_S_vs_aristas.png', dpi=150, bbox_inches='tight')
plt.close()
log("  fig4: S vs aristas — guardada")

# Figura 5: d_S vs g4 (Test 5 — el test más importante)
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
g4_arr = []
d_S_arr = []
for g4, data in test5_results.items():
    if 'd_S' in data:
        g4_arr.append(g4)
        d_S_arr.append(data['d_S'])
g4_arr = np.array(g4_arr)
d_S_arr = np.array(d_S_arr)

ax.plot(g4_arr, d_S_arr, 'o-', linewidth=2, markersize=10, label='d_S(meso) Hartree')
ax.axhline(2, color='purple', linestyle='--', label='ley de área (d=2, holográfico)')
ax.axhline(3, color='red', linestyle='--', label='ley de volumen (d=3, Λ exacta)')
ax.axhline(1, color='gray', linestyle=':', label='sub-área (d=1, no DE)')
ax.set_xlabel('g4 (anharmonicidad)')
ax.set_ylabel('d_S (banda mesoscópica)')
ax.set_title('¿La anharmonicidad cambia la dimensión efectiva del entrelazamiento?')
ax.set_xscale('symlog', linthresh=0.001)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig5_d_S_vs_g4.png', dpi=150, bbox_inches='tight')
plt.close()
log("  fig5: d_S vs g4 (Test 5) — guardada")

# Figura 6: Espectros vibracionales armónico vs anharmónico
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
omegas_arm_plot = np.array(test7_results['armonico']['omegas_full'])
omegas_arm_plot = omegas_arm_plot[omegas_arm_plot > 0]
ax.hist(omegas_arm_plot, bins=80, alpha=0.5, label='armónico (g4=0)', density=True)
for g4_key, data in test7_results['anharmonico'].items():
    if 'omegas_full' in data:
        omegas = np.array(data['omegas_full'])
        omegas = omegas[omegas > 0]
        ax.hist(omegas, bins=80, alpha=0.4, label=f'g4={g4_key}', density=True, histtype='step', linewidth=2)
ax.set_xlabel('ω (frecuencia modo)')
ax.set_ylabel('Densidad')
ax.set_title('Densidad espectral del Laplaciano: armónico vs anharmónico (Hartree)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/figuras/fig6_espectro.png', dpi=150, bbox_inches='tight')
plt.close()
log("  fig6: espectros — guardada")

log("Figuras guardadas.")

## 9. Reporte final y descarga automática

In [ ]:
# ============================================================
# Reporte final
# ============================================================

log("\n" + "="*70)
log("REPORTE FINAL")
log("="*70)

report = {
    'fecha': time.strftime('%Y-%m-%d %H:%M:%S'),
    'N_values': N_values,
    'exponentes_por_banda_y_N': {str(N): {b: {k: float(v) for k, v in d.items()} 
                                            for b, d in e.items()} 
                                  for N, e in exponents.items()},
    'extrapolacion_continuo': {},
    'test3_w_cosmologico': test3_results,
    'conclusiones': []
}

# Extrapolaciones
for band in ['UV', 'meso', 'IR']:
    Ns_avail = [N for N in N_values if band in exponents.get(N, {})]
    if len(Ns_avail) >= 2:
        d_S_arr = np.array([exponents[N][band]['d_S'] for N in Ns_avail])
        x_fit = 1.0 / np.array(Ns_avail)**(1/3)
        slope, intercept, r_val, _, _ = linregress(x_fit, d_S_arr)
        report['extrapolacion_continuo'][band] = {
            'd_S_inf': float(intercept),
            'd_S_por_N': {str(N): float(d) for N, d in zip(Ns_avail, d_S_arr)},
            'pendiente': float(slope),
            'R²': float(r_val**2),
            'w_predicho': float(-intercept / 3),
        }

# Conclusiones automáticas
if 'meso' in report['extrapolacion_continuo']:
    d_meso_inf = report['extrapolacion_continuo']['meso']['d_S_inf']
    w_meso = report['extrapolacion_continuo']['meso']['w_predicho']
    
    log(f"\n>>> Régimen mesoscópico (físicamente relevante):")
    log(f"    d_S(∞) = {d_meso_inf:.3f}")
    log(f"    w predicho = {w_meso:+.3f}")
    
    if abs(d_meso_inf - 2) < 0.15:
        report['conclusiones'].append("d_S meso ≈ 2: ley de área HOLOGRÁFICA confirmada")
        report['conclusiones'].append(f"w ≈ -2/3, energía oscura tipo quintessence")
    elif abs(d_meso_inf - 3) < 0.15:
        report['conclusiones'].append("d_S meso ≈ 3: ley de VOLUMEN")
        report['conclusiones'].append("w ≈ -1, Λ exacta")
    elif d_meso_inf < 1.5:
        report['conclusiones'].append(f"d_S meso = {d_meso_inf:.2f}: sub-área")
        report['conclusiones'].append("Sustrato NO es holográfico ni Λ; mecanismo armónico no produce DE")
    else:
        report['conclusiones'].append(f"d_S meso = {d_meso_inf:.2f}: régimen intermedio")
        report['conclusiones'].append(f"w predicho {w_meso:+.3f}: revisar interpretación")

# Resultados de Test 3
if abs(test3_results['w_homogeneo'] - 0.0698) < 0.01:
    report['conclusiones'].append("Test 3 control: w_homogéneo coincide con SIM 16/17")
if abs(test3_results['w_glassy'] - test3_results['w_homogeneo']) < 0.001:
    report['conclusiones'].append("Glassy NO modifica w en régimen armónico (ya esperado)")

ws_anh_arr = np.array(test3_results['ws_anharmonicos'])
if max(ws_anh_arr) - min(ws_anh_arr) > 0.01:
    report['conclusiones'].append(
        f"Anharmonicidad SÍ modifica w (rango {ws_anh_arr.min():+.3f} a {ws_anh_arr.max():+.3f})")
else:
    report['conclusiones'].append("Anharmonicidad perturbativa NO modifica w significativamente")

# === Conclusiones de Test 5 (escalado anharmónico de S) ===
if test5_results:
    d_S_g4_0 = test5_results.get(0.0, {}).get('d_S', None)
    d_S_g4_max = max([test5_results[g].get('d_S', d_S_g4_0) for g in test5_results if 'd_S' in test5_results[g]])
    if d_S_g4_0 is not None and abs(d_S_g4_max - d_S_g4_0) > 0.1:
        report['conclusiones'].append(
            f"Test 5: anharmonicidad SÍ modifica d_S(meso) (cambio {d_S_g4_max - d_S_g4_0:+.3f})")
        report['conclusiones'].append(
            "Esto significa que la anharmonicidad cambia el régimen holográfico — IMPORTANTE para Λ")
    else:
        report['conclusiones'].append(
            "Test 5: anharmonicidad NO modifica significativamente d_S(meso) en Hartree")
    report['test5_d_S_anharmonico'] = {str(g): test5_results[g].get('d_S', None) for g in test5_results}

# === Conclusiones de Test 6 (régimen no perturbativo) ===
if test6_results:
    valid_g4s = [r['g4'] for r in test6_results if r.get('valid', False)]
    if valid_g4s:
        report['conclusiones'].append(f"Test 6: Hartree válido hasta g4 ≈ {max(valid_g4s)}")
        if max(valid_g4s) < 1.0:
            report['conclusiones'].append(
                "Régimen no-perturbativo se necesita: usar DMRG/QMC/VMC en versión futura")
    report['test6_max_g4_perturbativo'] = max(valid_g4s) if valid_g4s else None

# === Conclusiones de Test 7 (espectro) ===
if test7_results:
    arm = test7_results['armonico']
    for g4 in [0.1, 1.0, 10.0]:
        if str(g4) in test7_results['anharmonico']:
            anh = test7_results['anharmonico'][str(g4)]
            delta_omega_min = anh['omega_min'] - arm['omega_min']
            if abs(delta_omega_min) > 0.01:
                report['conclusiones'].append(
                    f"Test 7: g4={g4} modifica modo IR significativamente (Δω={delta_omega_min:+.4f})")
                break
    report['test7_summary'] = {
        'omega_min_armonico': arm['omega_min'],
        'omega_min_g4_1': test7_results['anharmonico'].get('1.0', {}).get('omega_min', None),
    }

log("\n>>> Conclusiones automáticas:")
for c in report['conclusiones']:
    log(f"   - {c}")

# Guardar reporte
with open(f'{OUTPUT_DIR}/REPORTE_FINAL.json', 'w') as f:
    json.dump(report, f, indent=2)

# Reporte legible
with open(f'{OUTPUT_DIR}/REPORTE_FINAL.txt', 'w') as f:
    f.write(f"DEE — Test cosmológico grande\n")
    f.write(f"="*70 + "\n")
    f.write(f"Fecha: {report['fecha']}\n\n")
    
    f.write(f"Tamaños probados: N = {N_values}\n\n")
    
    f.write("Extrapolaciones al continuo:\n")
    for band, ext in report['extrapolacion_continuo'].items():
        f.write(f"  Banda {band}: d_S(∞) = {ext['d_S_inf']:.3f}, w = {ext['w_predicho']:+.3f}\n")
    
    f.write(f"\nTest 3 — w cosmológico:\n")
    f.write(f"  w homogéneo: {test3_results['w_homogeneo']:+.5f}\n")
    f.write(f"  w glassy:    {test3_results['w_glassy']:+.5f}\n")
    f.write(f"  w(g4) anharmónico: {test3_results['ws_anharmonicos']}\n")
    f.write(f"  w(g4) glassy+anh: {test3_results['ws_glassy_anharmonicos']}\n")
    
    f.write(f"\nTest 5 — escalado d_S anharmónico (banda meso):\n")
    for g4_key in sorted(test5_results.keys()):
        if 'd_S' in test5_results[g4_key]:
            f.write(f"  g4={g4_key}: d_S = {test5_results[g4_key]['d_S']:+.3f}\n")
    
    f.write(f"\nTest 6 — régimen no perturbativo:\n")
    valid_max = report.get('test6_max_g4_perturbativo', None)
    f.write(f"  Hartree válido hasta g4 ≈ {valid_max}\n")
    
    f.write(f"\nTest 7 — modo IR (relevante para Λ):\n")
    if 'test7_summary' in report:
        f.write(f"  ω_min armónico: {report['test7_summary']['omega_min_armonico']:.5f}\n")
        omega_g4_1 = report['test7_summary'].get('omega_min_g4_1')
        if omega_g4_1 is not None:
            f.write(f"  ω_min con g4=1: {omega_g4_1:.5f}\n")
    
    f.write("\nConclusiones:\n")
    for c in report['conclusiones']:
        f.write(f"  - {c}\n")

log("\nReporte final guardado en REPORTE_FINAL.json y REPORTE_FINAL.txt")
log_handle.close()

In [ ]:
# ============================================================
# Empaquetar todo y descargar
# ============================================================

import shutil

# Crear ZIP con todo
zip_path = '/content/dee_resultados.zip'
shutil.make_archive('/content/dee_resultados', 'zip', OUTPUT_DIR)

print(f"\nResultados empaquetados en: {zip_path}")
print(f"Tamaño: {os.path.getsize(zip_path)/1024:.1f} KB")

# Descarga automática
try:
    from google.colab import files
    print("\nIniciando descarga automática...")
    files.download(zip_path)
    print("Descarga iniciada.")
except ImportError:
    print("\nNo estamos en Colab. Para descargar manualmente:")
    print(f"  Buscar el archivo en: {zip_path}")

print("\n" + "="*70)
print("EJECUCIÓN COMPLETADA")
print("="*70)
print(f"Ver REPORTE_FINAL.txt para resumen.")